# Week 6 — Wednesday: Testing a Model — Splits and Metrics

**DATA 202 · Calvin University**

Monday, 2009's penguins showed how kNN does on **new** data. Usually there is no 2009 yet.

> 🎯 **How can we test a model on penguins it has never seen — before any new penguins exist?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~8 min | 1 · Hold some penguins out: the train/test split (SLO 06C) |
| ~6 min | 2 · Another split, another score |
| ~10 min | 3 · Where are the mistakes? The confusion matrix (SLO 06B) |
| ~10 min | 4 · Finding the Chinstraps: precision and recall (SLO 06B) |
| ~6 min | 5 · A score from one split |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

---
## 1 · Hold Some Penguins Out: the Train/Test Split (SLO 06C) · ~8 min

All **342 penguins**, 2007–2009. Pretend some of them are "next season":

| | share | used to… |
|---|---|---|
| **training set** | 70% | `fit` the model |
| **test set** | 30% | `predict` and score — never seen while learning |

The test set plays the part of Monday's 2009 penguins.

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

penguins = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/penguins.csv")
penguins = penguins.dropna(subset=["bill_length_mm", "bill_depth_mm"])

X = penguins[["bill_length_mm", "bill_depth_mm"]]
y = penguins["species"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=24)
print(len(X_train), "training penguins |", len(X_test), "test penguins")

* `test_size=0.3` → a random 30% goes to the test set
* `random_state=24` → the same "random" split every time we run it

In [ ]:
penguins["set"] = "training"
penguins.loc[X_test.index, "set"] = "test"

fig = px.scatter(penguins, x="bill_length_mm", y="bill_depth_mm", color="set", symbol="species",
                 color_discrete_map={"training": "lightgray", "test": "black"},
                 title="239 Training Penguins, 103 Test Penguins")
fig.show()

> 🎯 **Predict First:** Fit k = 1 and k = 5 on the **training** set only. What will each score on the training set? On the test set?

In [ ]:
for k in [1, 5]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)        # learns from the training set only
    print(f"k = {k} | train accuracy: {accuracy_score(y_train, knn.predict(X_train)):.3f}"
          f" | test accuracy: {accuracy_score(y_test, knn.predict(X_test)):.3f}")

<details><summary>Answer</summary>

| | train | test |
|---|---|---|
| k = 1 | **100%** | 94.2% |
| k = 5 | 99.2% | **95.1%** |

Monday's story again — without waiting for a new season. The **gap** between train and test is the sign of overfitting.
</details>

> 📌 **One split:** 95.1% is the score on *these* 103 test penguins.

---
## 2 · Another Split, Another Score · ~6 min

> 🎯 **Predict First:** Same model (k = 5), same penguins; only `random_state` changes. Same test accuracy?

In [ ]:
test_scores = []
for seed in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
    knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)
    test_scores.append(accuracy_score(y_te, knn.predict(X_te)))

splits = pd.DataFrame({"random_state": range(30), "test accuracy": test_scores})
splits["splits with this score"] = splits.groupby("test accuracy").cumcount() + 1   # stack equal scores
splits["which"] = splits["random_state"].map(lambda s: "our split (24)" if s == 24 else "another split")

fig = px.scatter(splits, x="test accuracy", y="splits with this score", color="which",
                 hover_name="random_state", title="k = 5, Test Accuracy on 30 Different Splits")
fig.update_traces(marker_size=14)
fig.show()
print("lowest:", round(min(test_scores), 3), "| highest:", round(max(test_scores), 3))

> 📄 **Handout 1 — 30 splits, 30 scores.** Write down the lowest and the highest score. Circle the split you'd be tempted to report. 🗣️ Why is that a problem?

<details><summary>Answer</summary>

* From **92.2%** to **100%** — same model, same penguins; only the luck of the split changed
* One split scores **100%**. Report only that one and you oversell the model
* **Week 7:** *cross-validation* — many splits, not one
</details>

---
## 3 · Where Are the Mistakes? The Confusion Matrix (SLO 06B) · ~10 min

Back to our split (`random_state=24`), k = 5: **95.1%** right. So 5 of the 103 test penguins were wrong. **Which ones?**

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
y_pred = knn.predict(X_test)

results = penguins.loc[X_test.index, ["species", "bill_length_mm", "bill_depth_mm"]]
results["predicted"] = y_pred
results[results["species"] != results["predicted"]]            # the 5 mistakes

> 📄 **Handout 2 — Build the confusion matrix.** Fill in the correct counts, then add the 5 mistakes. 🗣️ Compare with a neighbor.

In [ ]:
from sklearn.metrics import confusion_matrix

species = ["Adelie", "Chinstrap", "Gentoo"]
cm = pd.DataFrame(confusion_matrix(y_test, y_pred, labels=species), index=species, columns=species)

fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                labels=dict(x="predicted species", y="true species", color="penguins"),
                title="Confusion Matrix: 103 Test Penguins, k = 5")
fig.show()

**Reading it:** rows = **true** species · columns = **predicted** · the diagonal = **right**

<details><summary>Answer</summary>

* Diagonal: 46 + 18 + 34 = **98 right** → accuracy = 98 ÷ 103 = **95.1%**
* Every mistake involves a **Chinstrap**; Adelies and Gentoos are never confused
* Accuracy is one number. The matrix shows *where* the model fails
</details>

---
## 4 · Finding the Chinstraps: Precision and Recall (SLO 06B) · ~10 min

A team studying **Chinstraps**, the rarest species, asks one question: **Chinstrap or not?**

> 🎯 **Predict First:** A lazy "model" answers **not Chinstrap** for every penguin. What's its accuracy on this question?

In [ ]:
is_chinstrap = (y_test == "Chinstrap")               # the true answers: Chinstrap or not?
lazy = pd.Series(False, index=y_test.index)          # a "model" that never says Chinstrap

print("Chinstraps in the test set:", is_chinstrap.sum(), "of", len(y_test))
print("lazy model accuracy:", round(accuracy_score(is_chinstrap, lazy), 3))

**80.6% accurate — and it finds zero Chinstraps.** When a group is rare, accuracy can look fine while the model misses exactly what we care about.

Two questions accuracy doesn't ask:

| | asks | in the matrix |
|---|---|---|
| **Recall** | Of the **real** Chinstraps, how many did we find? | Chinstrap **row**: right ÷ row total |
| **Precision** | Of the penguins we **called** Chinstrap, how many really are? | Chinstrap **column**: right ÷ column total |

> 📄 **Handout 3 — Finding the Chinstraps.** From your matrix: recall, precision, and the lazy model's accuracy and recall.

In [ ]:
from sklearn.metrics import precision_score, recall_score

called_chinstrap = (y_pred == "Chinstrap")           # what kNN (k = 5) said
print("kNN  | recall:", round(recall_score(is_chinstrap, called_chinstrap), 3),
      "| precision:", round(precision_score(is_chinstrap, called_chinstrap), 3))
print("lazy | recall:", round(recall_score(is_chinstrap, lazy), 3))

<details><summary>Answer</summary>

* **Recall** = 18 ÷ 20 = **0.90** — we found 18 of the 20 Chinstraps; one was called an Adelie, one a Gentoo
* **Precision** = 18 ÷ 21 = **0.857** — 3 of the 21 "Chinstraps" were really 2 Adelies and a Gentoo
* **Lazy model:** recall = **0** — its 80.6% accuracy hid that
</details>

| mistake | name | lowers |
|---|---|---|
| called Chinstrap, but isn't | **false positive** | precision |
| a Chinstrap, but missed | **false negative** | recall |

> 📄 **Handout 3, last step:** write **FP** and **FN** in those cells of your matrix.

> 📌 **One split:** 0.90 and 0.857 describe *these* 20 Chinstraps. Another split, other numbers.

---
## 5 · A Score From One Split · ~6 min

Every number today came from **one split** of **342 penguins**, from **3 islands**, in **2007–2009**.

* Another split → another score: 92.2% to 100%
* Penguins from another colony, or a warmer decade → maybe very different
* Predictions work because the world is orderly — "seedtime and harvest … shall not cease" (Gen 8:22). But our data are partial: **generalizations are fallible**

A deep-learning model learned to spot pneumonia from about 160,000 chest X-rays, taken at three hospital sites:

> "While the system performed quite well, it turned out that a key reason for its success was that … the model was able to detect which hospital an X-ray originated with and adjust its predictions accordingly. … It learned, but not in the intended way."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** That model scored well when tested on X-rays from the *same* three sites. Why didn't the test catch the shortcut? What test would have?

<details><summary>Answer</summary>

* The test X-rays shared the training X-rays' quirks, so the shortcut worked there too: **a test set inherits the blind spots of the data it was split from**
* X-rays from a **new hospital** would have exposed it — just as our scores speak only for penguins like these
* With people, even the answer key can be a past human decision (who got hired, who got a loan). Then a good test score means "agrees with past decisions" — which is why validation gets "much more murky" (ch. 3)
</details>

---
### Wrap-up

1. **Split** — fit on the training set; score on a test set the model never saw
2. **Look inside the score** — the **confusion matrix** shows where the mistakes are
3. **Rare groups** — **recall** (did we find them?) and **precision** (were we right when we said so?)
4. 📌 Every score comes from **one split** of **these** penguins

**Week 7:** decision trees (another knob: depth) and cross-validation (many splits, not one).